## 1. Instalar yfinance

In [1]:
#!pip install yfinance

## 2. Imports y configuracion

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

TICKERS = ["NVDA", "GOOGL", "MSFT"]
PERIODO_INICIO = "2011-01-01"
PERIODO_FIN = "2025-12-31"
CARPETA_SALIDA = "datos"

os.makedirs(CARPETA_SALIDA, exist_ok=True)
print(f"Acciones: {TICKERS}")
print(f"Periodo: {PERIODO_INICIO} a {PERIODO_FIN}")

Acciones: ['NVDA', 'GOOGL', 'MSFT']
Periodo: 2011-01-01 a 2025-12-31


## 3. Funciones para indicadores tecnicos

In [3]:
def calcular_sma(close, periodo):
    """Simple Moving Average"""
    return close.rolling(window=periodo).mean()


def calcular_wma(close, periodo):
    """Weighted Moving Average"""
    pesos = np.arange(1, periodo + 1)
    return close.rolling(window=periodo).apply(
        lambda x: np.dot(x, pesos) / pesos.sum(), raw=True
    )


def calcular_rsi(close, periodo=14):
    """Relative Strength Index"""
    delta = close.diff()
    ganancia = delta.where(delta > 0, 0.0)
    perdida = -delta.where(delta < 0, 0.0)
    avg_ganancia = ganancia.rolling(window=periodo).mean()
    avg_perdida = perdida.rolling(window=periodo).mean()
    rs = avg_ganancia / avg_perdida
    return 100 - (100 / (1 + rs))


def calcular_macd(close):
    """MACD: EMA(12) - EMA(26)"""
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    return macd


def calcular_macd_signal(macd, periodo=9):
    """Linea de senal del MACD: EMA(9) del MACD"""
    return macd.ewm(span=periodo, adjust=False).mean()


def calcular_bollinger(close, periodo=20, num_std=2):
    """Bandas de Bollinger sobre SMA(20) +/- 2*sigma(20)"""
    media = close.rolling(window=periodo).mean()
    std = close.rolling(window=periodo).std()
    banda_superior = media + num_std * std
    banda_inferior = media - num_std * std
    return banda_superior, media, banda_inferior


def calcular_momentum(close, periodo=10):
    """Momentum porcentual: variacion relativa respecto al precio de hace N dias"""
    return close.pct_change(periodo)


def calcular_stochastic(high, low, close, k_periodo=14, d_periodo=3):
    """Stochastic K y D"""
    min_low = low.rolling(window=k_periodo).min()
    max_high = high.rolling(window=k_periodo).max()
    k = 100 * (close - min_low) / (max_high - min_low)
    d = k.rolling(window=d_periodo).mean()
    return k, d


def calcular_williams_r(high, low, close, periodo=14):
    """Williams %R (WPR): -100 * (HH - Close) / (HH - LL), acotado [-100, 0]"""
    max_high = high.rolling(window=periodo).max()
    min_low = low.rolling(window=periodo).min()
    return -100 * (max_high - close) / (max_high - min_low)


print("Funciones cargadas.")

Funciones cargadas.


## 4. Descargar y procesar cada accion

In [4]:
def descargar_contexto_mercado(inicio, fin):
    """
    Descarga 3 features externas de contexto de mercado (Hu y Luo, 2023):
      - SP500_ret  : retorno diario S&P 500 (^GSPC)   — benchmark general
      - NASDAQ_ret : retorno diario NASDAQ 100 (^NDX)  — índice tecnológico
      - Gold_ret   : retorno diario Oro (GLD)          — activo refugio
    """
    print("\n[CONTEXTO MERCADO] Descargando S&P 500, NASDAQ y Oro...")

    activos = [("^GSPC", "SP500_ret"), ("^NDX", "NASDAQ_ret"), ("GLD", "Gold_ret")]
    df = pd.DataFrame()

    for ticker, col in activos:
        datos = yf.Ticker(ticker).history(start=inicio, end=fin, auto_adjust=True)
        if datos.index.tz is not None:
            datos.index = datos.index.tz_convert(None)
        datos.index = datos.index.normalize()
        df[col] = datos["Close"].pct_change()
        print(f"  {col}: {datos['Close'].notna().sum()} filas descargadas")

    print(f"  Columnas: {list(df.columns)}")
    return df


def procesar_accion(ticker, df_contexto):
    print(f"\n{'='*60}")
    print(f"Procesando {ticker}...")
    print(f"{'='*60}")

    # --- 1. Descargar OHLCV ---
    print(f"[1/4] Descargando datos OHLCV...")
    accion = yf.Ticker(ticker)
    df = accion.history(start=PERIODO_INICIO, end=PERIODO_FIN, auto_adjust=True)

    if df.empty:
        print(f"ERROR: No se encontraron datos para {ticker}")
        return None

    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)
    else:
        df.index = df.index.tz_localize(None)
    df.index = df.index.normalize()
    df.index.name = "Date"

    print(f"  Filas descargadas: {len(df)}")
    print(f"  Rango: {df.index[0].strftime('%Y-%m-%d')} a {df.index[-1].strftime('%Y-%m-%d')}")

    # --- 2. Calcular indicadores tecnicos ---
    print(f"[2/4] Calculando indicadores tecnicos...")

    df["SMA_20"] = calcular_sma(df["Close"], 20)
    df["SMA_50"] = calcular_sma(df["Close"], 50)
    df["WMA_20"] = calcular_wma(df["Close"], 20)
    df["RSI_14"] = calcular_rsi(df["Close"], 14)
    df["MACD"] = calcular_macd(df["Close"])
    df["MACD_signal"] = calcular_macd_signal(df["MACD"], 9)
    df["BB_upper"], df["BB_mid"], df["BB_lower"] = calcular_bollinger(df["Close"], 20, 2)
    df["Momentum_10"] = calcular_momentum(df["Close"], 10)
    df["Stoch_K"], df["Stoch_D"] = calcular_stochastic(df["High"], df["Low"], df["Close"])
    df["WilliamsR_14"] = calcular_williams_r(df["High"], df["Low"], df["Close"], 14)

    print(f"  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D, WilliamsR_14")

    # --- 3. Agregar features externas ---
    print(f"[3/4] Agregando S&P 500, NASDAQ y Oro...")
    df = df.join(df_contexto, how="left")
    features_ext = ["SP500_ret", "NASDAQ_ret", "Gold_ret"]
    nans_ext = df[features_ext].isnull().sum()
    if nans_ext.any():
        print(f"  NaN en primeras filas (esperado): {nans_ext[nans_ext > 0].to_dict()}")

    # --- 4. Variable objetivo ---
    print(f"[4/4] Calculando variable objetivo...")
    df["Target_Retorno_1d"] = df["Close"].pct_change(1).shift(-1)
    df["Target_Direccion"] = (df["Target_Retorno_1d"] > 0).astype(int)

    # Eliminar filas con NaN
    filas_antes = len(df)
    df = df.dropna()
    print(f"  Filas eliminadas por NaN: {filas_antes - len(df)}")
    print(f"  Filas finales: {len(df)}")

    return df

In [5]:
df_contexto = descargar_contexto_mercado(PERIODO_INICIO, PERIODO_FIN)

resultados = {}
for ticker in TICKERS:
    df = procesar_accion(ticker, df_contexto)
    if df is not None:
        resultados[ticker] = df


[CONTEXTO MERCADO] Descargando S&P 500, NASDAQ y Oro...
  SP500_ret: 3771 filas descargadas
  NASDAQ_ret: 3771 filas descargadas
  Gold_ret: 3771 filas descargadas
  Columnas: ['SP500_ret', 'NASDAQ_ret', 'Gold_ret']

Procesando NVDA...
[1/4] Descargando datos OHLCV...
  Filas descargadas: 3771
  Rango: 2011-01-03 a 2025-12-30
[2/4] Calculando indicadores tecnicos...
  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D, WilliamsR_14
[3/4] Agregando S&P 500, NASDAQ y Oro...
  NaN en primeras filas (esperado): {'SP500_ret': 1, 'NASDAQ_ret': 1, 'Gold_ret': 1}
[4/4] Calculando variable objetivo...
  Filas eliminadas por NaN: 50
  Filas finales: 3721

Procesando GOOGL...
[1/4] Descargando datos OHLCV...
  Filas descargadas: 3771
  Rango: 2011-01-03 a 2025-12-30
[2/4] Calculando indicadores tecnicos...
  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D, WilliamsR_14
[3/4] Agr

## 5. Guardar CSVs

In [7]:
carpeta_abs = os.path.abspath(CARPETA_SALIDA)
os.makedirs(carpeta_abs, exist_ok=True)

for ticker, df in resultados.items():
    archivo = os.path.join(carpeta_abs, f"{ticker}_dataset.csv")
    df.to_csv(archivo)
    tamano = os.path.getsize(archivo) / 1024
    print(f"{ticker}_dataset.csv  ->  {tamano:.1f} KB  ({len(df)} filas x {len(df.columns)} columnas)")

print(f"\nCarpeta: {carpeta_abs}")

NVDA_dataset.csv  ->  1570.9 KB  (3721 filas x 23 columnas)
GOOGL_dataset.csv  ->  1553.3 KB  (3721 filas x 23 columnas)
MSFT_dataset.csv  ->  1548.7 KB  (3721 filas x 23 columnas)

Carpeta: c:\Users\hamga\Documents\repo\ai-stock-price-prediction\datos


## 6. Resumen y verificacion

In [8]:
print(f"{'='*60}")
print("RESUMEN FINAL")
print(f"{'='*60}")

for ticker, df in resultados.items():
    print(f"\n--- {ticker} ---")
    print(f"  Filas: {len(df)} (sin NaN)")
    print(f"  Columnas ({len(df.columns)}):")
    for col in df.columns:
        print(f"    - {col}: {df[col].notna().sum()} valores")

RESUMEN FINAL

--- NVDA ---
  Filas: 3721 (sin NaN)
  Columnas (23):
    - Open: 3721 valores
    - High: 3721 valores
    - Low: 3721 valores
    - Close: 3721 valores
    - Volume: 3721 valores
    - SMA_20: 3721 valores
    - SMA_50: 3721 valores
    - WMA_20: 3721 valores
    - RSI_14: 3721 valores
    - MACD: 3721 valores
    - MACD_signal: 3721 valores
    - BB_upper: 3721 valores
    - BB_mid: 3721 valores
    - BB_lower: 3721 valores
    - Momentum_10: 3721 valores
    - Stoch_K: 3721 valores
    - Stoch_D: 3721 valores
    - WilliamsR_14: 3721 valores
    - SP500_ret: 3721 valores
    - NASDAQ_ret: 3721 valores
    - Gold_ret: 3721 valores
    - Target_Retorno_1d: 3721 valores
    - Target_Direccion: 3721 valores

--- GOOGL ---
  Filas: 3721 (sin NaN)
  Columnas (23):
    - Open: 3721 valores
    - High: 3721 valores
    - Low: 3721 valores
    - Close: 3721 valores
    - Volume: 3721 valores
    - SMA_20: 3721 valores
    - SMA_50: 3721 valores
    - WMA_20: 3721 valores
    

## 7. Vista previa

In [9]:
for ticker, df in resultados.items():
    print(f"\n{'='*40}")
    print(f"{ticker} - Primeras 5 filas")
    print(f"{'='*40}")
    display(df.head())


NVDA - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15,0.396739,0.411636,0.389863,0.404760,1256280000,0.488199,0.505387,0.461089,19.394664,-0.028884,...,0.378268,-0.184296,9.999996,8.215778,-90.000004,-0.011200,-0.013576,-0.018652,-0.007361,0
2011-03-16,0.401094,0.420575,0.396510,0.401781,1476520000,0.482447,0.506171,0.452859,12.682052,-0.030533,...,0.366925,-0.155181,8.000161,9.076947,-91.999839,-0.019495,-0.025071,-0.000220,0.018824,1
2011-03-17,0.412553,0.413241,0.394218,0.409344,1238516000,0.476121,0.507129,0.445897,10.862825,-0.030874,...,0.359069,-0.144226,13.076977,10.359045,-86.923023,0.013398,0.010109,0.005358,-0.013438,0
2011-03-18,0.415762,0.417137,0.403385,0.403844,886960000,0.466884,0.507422,0.439013,11.230488,-0.031227,...,0.358352,-0.151252,10.200761,10.425966,-89.799239,0.004310,-0.001874,0.010221,0.007945,1
2011-03-21,0.412553,0.416679,0.402927,0.407052,751832000,0.457865,0.506702,0.433315,15.452578,-0.030893,...,0.362308,-0.132389,14.940238,12.739325,-85.059762,0.014986,0.018743,0.005565,-0.017455,0



GOOGL - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15,13.837088,14.172157,13.787447,14.136415,160063776,14.943472,15.146590,14.713761,20.094535,-0.266250,...,14.042143,-0.051934,22.065301,10.821106,-77.934699,-0.011200,-0.013576,-0.018652,-0.021877,0
2011-03-16,14.097944,14.142125,13.682709,13.827159,151788060,14.860263,15.123136,14.607445,17.585842,-0.308382,...,13.869103,-0.072721,8.566295,11.493082,-91.433705,-0.019495,-0.025071,-0.000220,0.007647,1
2011-03-17,14.010330,14.122515,13.912540,13.932892,115856028,14.782254,15.102902,14.519124,20.610958,-0.329442,...,13.755839,-0.079073,14.836544,15.156047,-85.163456,0.013398,0.010109,0.005358,-0.000534,0
2011-03-18,14.014301,14.097449,13.892685,13.925447,131812056,14.702582,15.079070,14.437523,17.189060,-0.342782,...,13.669569,-0.065865,14.395016,12.599285,-85.604984,0.004310,-0.001874,0.010221,0.027519,1
2011-03-21,14.152798,14.390573,14.123014,14.308666,120715164,14.636090,15.060704,14.400008,35.307672,-0.318757,...,13.689120,-0.025623,41.886751,23.706104,-58.113249,0.014986,0.018743,0.005565,0.001422,1



MSFT - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15,19.161321,19.459283,19.100200,19.398163,76067300,20.070872,20.850545,19.848528,28.571569,-0.393891,...,19.204643,-0.029434,18.931980,19.224295,-81.068020,-0.011200,-0.013576,-0.018652,-0.023631,0
2011-03-16,19.268288,19.314130,18.855725,18.939766,100725400,19.987978,20.804313,19.740803,19.254676,-0.432769,...,19.022396,-0.049463,4.845855,14.553743,-95.154145,-0.019495,-0.025071,-0.000220,-0.000404,0
2011-03-17,19.146041,19.268282,18.909199,18.932119,62497000,19.902409,20.756258,19.640245,20.597983,-0.458906,...,18.879786,-0.054199,4.586835,9.454890,-95.413165,0.013398,0.010109,0.005358,0.000807,1
2011-03-18,19.146043,19.237725,18.947401,18.947401,85486700,19.810346,20.709876,19.549292,20.333681,-0.472936,...,18.792150,-0.044316,5.714034,5.048908,-94.285966,0.004310,-0.001874,0.010221,0.021371,1
2011-03-21,19.237723,19.543326,19.214802,19.352324,46878100,19.744259,20.659136,19.505671,36.656071,-0.446237,...,18.792710,-0.015163,37.790284,16.030384,-62.209716,0.014986,0.018743,0.005565,-0.001184,0


In [10]:
for ticker, df in resultados.items():
    print(f"\n{'='*40}")
    print(f"{ticker} - Estadisticas descriptivas")
    print(f"{'='*40}")
    display(df.describe())


NVDA - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,24.385390,24.795082,23.926178,24.383378,4.496793e+08,23.917939,23.175550,24.071357,55.239183,0.343081,...,21.563399,0.019921,58.440300,58.420008,-41.559700,0.000509,0.000736,0.000333,0.002056,0.529965
std,45.140213,45.826284,44.321389,45.110543,2.619554e+08,44.340014,43.062459,44.581360,16.856557,1.504818,...,40.488095,0.086606,30.298307,28.205228,30.298307,0.010924,0.013225,0.009949,0.028573,0.499168
min,0.264034,0.266555,0.255554,0.260825,4.564400e+07,0.273492,0.280945,0.274471,2.978679,-6.256588,...,0.255552,-0.323121,0.000000,0.946958,-100.000000,-0.119841,-0.121932,-0.087808,-0.187559,0.000000
25%,0.472349,0.479277,0.466764,0.471708,2.769320e+08,0.475719,0.471339,0.476509,42.856984,-0.005301,...,0.445586,-0.030530,32.098726,33.131218,-67.901274,-0.003779,-0.004880,-0.004804,-0.011949,0.000000
50%,4.573795,4.679530,4.504638,4.584127,3.936940e+08,4.509900,4.380964,4.514283,55.197213,0.011755,...,4.149729,0.017314,63.408083,62.948852,-36.591917,0.000688,0.001157,0.000505,0.001832,1.000000
75%,19.597110,19.908756,19.129813,19.556395,5.486480e+08,19.513375,18.684765,19.501943,67.561481,0.191536,...,16.818167,0.069137,85.960670,84.214069,-14.039330,0.005675,0.007415,0.005549,0.016283,1.000000
max,208.057150,212.166717,205.537422,207.017273,3.692928e+09,193.373268,187.287456,194.875129,99.428670,9.771465,...,177.542753,0.391485,100.000000,99.205180,-0.000000,0.095154,0.120223,0.049038,0.298066,1.000000



GOOGL - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,74.642555,75.446640,73.855722,74.672925,4.940828e+07,73.917727,72.764406,74.169511,54.738150,0.541228,...,69.553714,0.009523,57.954671,57.937358,-42.045329,0.000509,0.000736,0.000333,0.000986,0.532384
std,58.833994,59.536376,58.142199,58.858734,3.997943e+07,57.625616,55.742213,58.028785,16.385218,2.086111,...,53.676161,0.051377,30.059529,27.859548,30.059529,0.010924,0.013225,0.009949,0.017490,0.499017
min,11.764628,11.932162,11.740303,11.786469,9.312000e+06,12.419947,12.858574,12.259562,10.962637,-7.034280,...,11.133080,-0.226008,0.000000,0.537580,-100.000000,-0.119841,-0.121932,-0.087808,-0.116341,0.000000
25%,28.060083,28.177614,27.728035,27.985699,2.675400e+07,27.844218,27.767117,27.896387,42.407879,-0.179855,...,26.727813,-0.019771,32.447714,33.640381,-67.552286,-0.003779,-0.004880,-0.004804,-0.007431,0.000000
50%,54.549148,55.213660,53.876214,54.552631,3.512600e+07,54.527977,54.445598,54.410862,55.026570,0.269208,...,51.190049,0.009234,62.410173,61.984728,-37.589827,0.000688,0.001157,0.000505,0.001024,1.000000
75%,114.761493,116.347880,113.370987,114.883987,5.814600e+07,113.908894,112.667493,113.833307,66.522413,1.006930,...,105.693773,0.040255,85.577600,83.876981,-14.422400,0.005675,0.007415,0.005549,0.009651,1.000000
max,325.767421,328.383862,318.736980,323.001190,5.611863e+08,313.436005,293.921053,313.874727,98.093686,14.457856,...,300.025599,0.283183,100.000000,98.982432,-0.000000,0.095154,0.120223,0.049038,0.162584,1.000000



MSFT - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_lower,Momentum_10,Stoch_K,Stoch_D,WilliamsR_14,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,159.152857,160.623845,157.611485,159.190288,3.307108e+07,158.011110,156.122301,158.402906,55.483606,0.876801,...,150.238903,0.009563,59.999285,59.983816,-40.000715,0.000509,0.000736,0.000333,0.000998,0.526471
std,145.519265,146.749743,144.176513,145.515157,1.855777e+07,144.659188,143.214890,144.927049,15.688838,3.311776,...,137.963104,0.043643,29.395027,27.069553,29.395027,0.010924,0.013225,0.009949,0.016250,0.499366
min,18.264127,18.464082,18.187225,18.233366,5.855900e+06,18.607108,19.087470,18.532478,6.806019,-11.641846,...,17.693923,-0.216274,0.000000,1.988610,-100.000000,-0.119841,-0.121932,-0.087808,-0.147390,0.000000
25%,37.432515,37.767655,37.172748,37.484467,2.148710e+07,37.423423,37.532140,37.507711,44.236209,-0.170183,...,35.231522,-0.014506,36.510561,37.866536,-63.489439,-0.003779,-0.004880,-0.004804,-0.006887,0.000000
50%,95.981614,96.811589,94.407379,95.758057,2.810730e+07,95.285783,94.677036,95.314229,55.334906,0.378012,...,90.149647,0.010438,65.784971,65.720418,-34.215029,0.000688,0.001157,0.000505,0.000725,1.000000
75%,265.740044,268.696698,261.517376,265.665222,3.892310e+07,260.198662,258.657432,261.177917,67.183965,1.630806,...,241.381599,0.035203,86.096102,84.224533,-13.903898,0.005675,0.007415,0.005549,0.009227,1.000000
max,550.830186,551.048474,537.366763,538.658569,3.193179e+08,518.459973,511.207405,520.604707,99.112156,18.555757,...,505.188574,0.215399,100.000000,98.881178,-0.000000,0.095154,0.120223,0.049038,0.142169,1.000000
